In [54]:
import os
import plotly.graph_objects as go
import pandas as pd
import torch
import numpy as np
from collections import Counter

In [9]:
figures_directory = "figures"
os.makedirs(figures_directory, exist_ok=True)

In [10]:
overall_character_dataset_path = os.path.join("data", "overall_character_dataset.pt")

# ✅ Load it back
overall_character_dataset = torch.load(overall_character_dataset_path)

print(f"Character count: {len(overall_character_dataset)}")
print(f"Character count: {overall_character_dataset[0].keys()}")

Character count: 29849
Character count: dict_keys(['file_name', 'char_name', 'char_id', 'mention_count', 'mention_ratio', 'attributes_count', 'plural_ratio', 'gender_ratio', 'agent_lemmas', 'patient_lemmas', 'mod_lemmas', 'pos_lemmas', 'mean_embedding', 'label', 'year', 'author', 'multidoc_alias'])


In [16]:
keys_to_keep = ["year", "label", "mentions", "mention_ratio"]
chapitre_characters_df = pd.DataFrame([{k: d[k] for k in keys_to_keep if k in d} for d in overall_character_dataset])

In [17]:
# Prepare data
mean_per_year = chapitre_characters_df.groupby('year')['label'].mean().reset_index()
rolling_mean_window = 15
mean_per_year['rolling_mean'] = mean_per_year['label'].rolling(window=rolling_mean_window, center=True).mean()

# Convert to percentage
mean_per_year['label'] *= 100
mean_per_year['rolling_mean'] *= 100


# Create figure
fig = go.Figure()

# Add scatter plot (points) for mean probabilities
fig.add_trace(go.Scatter(
    x=mean_per_year['year'],
    y=mean_per_year['label'],
    mode='markers',
    name='Mean per Year',
    marker=dict(color='#1f77b4', size=8)
))

# Add line plot for rolling mean
fig.add_trace(go.Scatter(
    x=mean_per_year['year'],
    y=mean_per_year['rolling_mean'],
    mode='lines',
    name=f'{rolling_mean_window}-Year Rolling Mean',
    line=dict(color='#ff7f0e', width=4)
))

figure_aspect_ratio = 3
width = 1200
height=width/figure_aspect_ratio

# Update layout
fig.update_layout(
    # title='Predicted Probability by Year',
    xaxis_title='Year',
    yaxis_title='Predicted Detective Ratio (%)',
    xaxis=dict(
        range=[
            chapitre_characters_df["year"].min() + 5 ,
            chapitre_characters_df["year"].max() -3.5,
        ],
        showgrid=False,
        gridcolor='lightgray',  # or 'gray', '#CCCCCC', etc.
        gridwidth=1,
        showticklabels=True,
        dtick=20,           # Set tick interval (e.g., every 10 years)
        ticks='inside',     # place ticks outside the axis
        tickcolor='black',   # tick color
        tickwidth=2,         # tick line thickness
        ticklen=10            # length of tick lines
    ),
    yaxis=dict(
        range=[-0.5, 14.25],
        showgrid=True,
        dtick=2,
        # tickcolor='black',
        gridcolor='lightgray',
        gridwidth=1
    ),
    height=height,
    width=width,
    legend=dict(
        x=0.1,         # Horizontal position (0 = left, 1 = right)
        y=0.75,         # Vertical position (0 = bottom, 1 = top)
        xanchor='left', # Anchor legend box at left of x
        yanchor='top',  # Anchor legend box at top of y
        font=dict(
            family='Times Roman',
            size=22,
            color='black'
        ),
        bgcolor='white',     # Optional: legend background color
        bordercolor='grey', # Optional: legend border color
        borderwidth=0.25
    ),
    font=dict(
        family="Times Roman",
        size=22,
        color="black"
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(
        l=100,# left
        r=30,   # right
        t=10,   # top
        b=0    # bottom
    )
)

fig.show()
fig.write_image(f"{os.path.join(figures_directory, 'predicted_detective_ratio.png')}",
                width=width,
                height=height,
                scale=4)

In [18]:
predicted_detectives_df = chapitre_characters_df[chapitre_characters_df['label'] == 1].copy().reset_index(drop=True)

predicted_detectives_df = predicted_detectives_df.sort_values(by=["year"], ascending=True)

x = predicted_detectives_df['year']
y = predicted_detectives_df['mention_ratio']

coeffs = np.polyfit(x, y, deg=5)
trendline = np.poly1d(coeffs)

time_interval = 1
predicted_detectives_df["year"] = predicted_detectives_df["year"].apply(lambda x : int(x // time_interval * time_interval + time_interval/2))
predicted_detectives_df = predicted_detectives_df.groupby('year')['mention_ratio'].mean().reset_index()

# predicted_detectives_df.loc[len(predicted_detectives_df), ["year", "mention_ratio"]] = [
#     chapitre_characters_df["year"].min(),
#     0]


predicted_detectives_df = predicted_detectives_df.sort_values("year")
predicted_detectives_df['trendline'] = trendline(predicted_detectives_df['year'])

In [20]:
fig = go.Figure()

# Add scatter plot (points) for mean probabilities
fig.add_trace(go.Scatter(
    x=predicted_detectives_df['year'],
    y=predicted_detectives_df['mention_ratio'],
    mode='markers',
    # hovertext=predicted_detectives_df['author_character'],
    name='Predicted Detective Mention Ratio - Mean per Year',
    marker=dict(color='#1f77b4', size=8)
))

# Add line plot for rolling mean
fig.add_trace(go.Scatter(
    x=predicted_detectives_df['year'],
    y=predicted_detectives_df['trendline'],
    mode='lines',
    name=f'Trendline',
    line=dict(color='#ff7f0e', width=5, dash='dash'),
))

figure_aspect_ratio = 2.5
width = 1200
height=width/figure_aspect_ratio

# Update layout
fig.update_layout(
    # title='Predicted Probability by Year',
    xaxis_title='Year',
    yaxis_title='Mention ratio of predicted detectives',
    xaxis=dict(
        range=[
            1855,
            2021
        ],
        showgrid=False,
        gridcolor='lightgray',  # or 'gray', '#CCCCCC', etc.
        gridwidth=1,
        showticklabels=True,
        dtick=20,           # Set tick interval (e.g., every 10 years)
        ticks='inside',     # place ticks outside the axis
        tickcolor='black',   # tick color
        tickwidth=2,         # tick line thickness
        ticklen=10            # length of tick lines
    ),
    yaxis=dict(
        range=[-0.01, 0.45],
        showgrid=True,
        # dtick=0.025,
        # tickcolor='black',
        gridcolor='lightgray',
        gridwidth=1
    ),
    height=height,
    width=width,
    legend=dict(
        x=0.01,         # Horizontal position (0 = left, 1 = right)
        y=0.99,         # Vertical position (0 = bottom, 1 = top)
        xanchor='left', # Anchor legend box at left of x
        yanchor='top',  # Anchor legend box at top of y
        font=dict(
            family='Times Roman',
            size=22,
            color='black'
        ),
        bgcolor='white',     # Optional: legend background color
        bordercolor='grey', # Optional: legend border color
        borderwidth=0.25
    ),
    font=dict(
        family="Times Roman",
        size=22,
        color="black"
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(
        l=100,# left
        r=20,   # right
        t=10,   # top
        b=0    # bottom
    )
)

fig.show()
fig.write_image(f"{os.path.join(figures_directory, 'mention_ratio_evolution.png')}",
                width=width,
                height=height,
                scale=4)

In [37]:
keys_to_keep = ["year", "label", "mentions", "mention_ratio", "author", "multidoc_alias"]
chapitre_characters_df = pd.DataFrame([{k: d[k] for k in keys_to_keep if k in d} for d in overall_character_dataset])

predicted_detectives_df = chapitre_characters_df[chapitre_characters_df["label"] == 1].copy()
detective_embeddings = [char_dict["mean_embedding"] for char_dict in overall_character_dataset if char_dict["label"] ==1]

In [38]:
len(predicted_detectives_df)

713

In [44]:
import numpy as np
import umap.umap_ as umap  # make sure umap-learn is installed

# Convert list of arrays to 2D numpy array
X = np.stack(detective_embeddings)  # shape: (29865, 1024)
X = np.nan_to_num(X, nan=0.0)

# Initialize UMAP
reducer = umap.UMAP(
    n_neighbors=10,
    n_components=2,
    min_dist=0.14,
    metric='cosine',
    random_state=42,
    spread=1.0,
    learning_rate=1.0,
    init='spectral',
)


# Fit and transform
embedding_2d = reducer.fit_transform(X)  # shape: (29865, 2)

# Extract x, y coordinates
x_coords = embedding_2d[:, 0]
y_coords = embedding_2d[:, 1]


from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3,
                random_state=42
                )
labels = kmeans.fit_predict(embedding_2d)

from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

# Get distances to closest center
closest_center, distances = pairwise_distances_argmin_min(embedding_2d, kmeans.cluster_centers_)

# Mark points as "outliers" if their distance exceeds a threshold (e.g., 95th percentile)
threshold = np.percentile(distances, 100)
clusters_labels = np.where(distances > threshold, -1, labels)

predicted_detectives_df["umap_x"] = x_coords
predicted_detectives_df["umap_y"] = y_coords
predicted_detectives_df["cluster"] = clusters_labels

/home/antoine/Bureau/character_attributes_classification/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/home/antoine/Bureau/character_attributes_classification/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [40]:
import plotly.express as px


for column in [
    # "mention_count",
               # "attributes_count",
               # "gender_ratio",
               "author",
               "multidoc_alias",
               "year",
               # "multi_doc_character",
               # "cluster"
]:
# for column in ["year"]:
    fig = px.scatter(
        predicted_detectives_df,
        x="umap_x",
        y="umap_y",
        color=column,
        hover_data=[column for column in predicted_detectives_df.columns if 'att' not in column],  # show all columns on hover; customize as needed
        title=f"UMAP Projection of Embeddings - Color='{column}'",
        width=1250,
        height=750,
        color_continuous_scale=px.colors.sequential.Jet,
    )
    fig.update_layout(
        margin=dict(
        l=0,# left
        r=0,   # right
        t=25,   # top
        b=0    # bottom
    ))

    fig.show()

    fig.write_image(f"{figures_directory}/UMAP_projection_{column}.png",
                    width=width,
                    height=height,
                    scale=4)

In [42]:
import plotly.graph_objects as go
from scipy.spatial import ConvexHull
import numpy as np
from scipy.spatial import ConvexHull
from scipy.interpolate import splprep, splev

width=1250
height=700

fig = px.scatter(
    predicted_detectives_df,
    x="umap_x",
    y="umap_y",
    color="year",
    hover_data=[col for col in predicted_detectives_df.columns if 'att' not in col],
    title=None,
    width=width,
    height=height,
    color_continuous_scale=px.colors.sequential.Jet,
)


fig.update_traces(
    marker=dict(
        line=dict(
            width=0.5,         # Scatter point border width
            color="black"    # Scatter point border color
        ),
        size=10               # Optional: make points bigger
    )
)

# Draw smooth convex hull for each cluster
for cluster_id in predicted_detectives_df["cluster"].unique():
    cluster_points = predicted_detectives_df[predicted_detectives_df["cluster"] == cluster_id][["umap_x", "umap_y"]].values

    if len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        hull_points = cluster_points[hull.vertices]
        hull_points = np.vstack([hull_points, hull_points[0]])  # close the loop

        # Interpolate a spline around the hull points
        x, y = hull_points[:, 0], hull_points[:, 1]
        tck, _ = splprep([x, y], s=0.05, per=True)  # `s` is smoothness, tweak as needed
        interp_points = splev(np.linspace(0, 1, 180), tck)

        fig.add_trace(
            go.Scatter(
                x=interp_points[0],
                y=interp_points[1],
                mode='lines',
                line=dict(color='black', width=1, dash='dash'),
                name=f"Cluster {cluster_id} boundary",
                showlegend=False,
            )
        )

# Cluster names and manual (dx, dy) offsets
cluster_labels = {
    0: {"name": "Cluster A", "offset": (-1, 2)},
    1: {"name": "Cluster C", "offset": (1.6, -1.4)},
    2: {"name": "Cluster B", "offset": (0, -1.1)},
    # Add more as needed
}

for cluster_id in predicted_detectives_df["cluster"].unique():
    cluster_points = predicted_detectives_df[predicted_detectives_df["cluster"] == cluster_id][["umap_x", "umap_y"]].values
    centroid = np.mean(cluster_points, axis=0)

    # Apply manual offset
    offset = cluster_labels.get(cluster_id, {}).get("offset", (0, 0))
    label_x = centroid[0] + offset[0]
    label_y = centroid[1] + offset[1]
    label_text = cluster_labels.get(cluster_id, {}).get("name", f"Cluster {cluster_id}")

    fig.add_trace(go.Scatter(
        x=[label_x],
        y=[label_y],
        text=[label_text],
        mode="text",
        showlegend=False,
        textfont=dict(color="black", size=32, family="Times Roman", weight=10),
    ))


# White background and remove grid/zero lines
fig.update_layout(
    autosize=False,
    width=width,
    height=height,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=0, r=0, t=0, b=0),
    coloraxis_colorbar=dict(
        title=dict(
            text="Publication Year",
            font=dict(color="black", size=20)
        ),
        tickfont=dict(color="black", size=20)
    ),
)

fig.update_xaxes(showgrid=False, showticklabels=False, zeroline=False)
fig.update_yaxes(showgrid=False, showticklabels=False, zeroline=False)

fig.show()
fig.write_image(f"figures/detective_clusters.png",
                    width=width,
                    height=height,
                    scale=4)
fig.write_html("figures/detective_clusters.html")

## Discriminative attributes

In [50]:
column = "mod_lemmas"
attributes_lemme_correction_mapping_dict = {
    "train": "-",
    "mesure": "en mesure de",
    "courant": "au courant de",
    "priver": "privé",
    "bel": "beau",
    "marier": "marié"
}
for char_dict in overall_character_dataset:
    char_dict[column] = [
        attributes_lemme_correction_mapping_dict.get(v, v)
        for v in char_dict[column]
    ]

column = "pos_lemmas"
attributes_lemme_correction_mapping_dict = {
    "m.": "monsieur",
    "mme.": "madame",
    "mme": "madame",
    "juve": "-",
    "fandor": "-"
}
for char_dict in overall_character_dataset:
    char_dict[column] = [
        attributes_lemme_correction_mapping_dict.get(v, v)
        for v in char_dict[column]
    ]

column = "agent_lemmas"
attributes_lemme_correction_mapping_dict = {
    "demande": "demander",
    "souvenir": "se souvenir",
    "écrier": "s'écrier",
    "porte": "porter",
    "voilà": "-",
    "être": "-",
    "faillir": "-",
}
for char_dict in overall_character_dataset:
    char_dict[column] = [
        attributes_lemme_correction_mapping_dict.get(v, v)
        for v in char_dict[column]
    ]

In [79]:
def get_z_score_df(characters_dict_A, character_dicts_B, attributes_type="mod_lemmas"):
    A_attributes = [item for char_dict in characters_dict_A for item in char_dict[attributes_type]]
    A_attributes = [item for item in A_attributes if item != "-"]

    B_attributes = [item for char_dict in character_dicts_B for item in char_dict[attributes_type]]
    B_attributes = [item for item in B_attributes if item != "-"]

    # Count all attributes
    attr_counts_A = Counter(A_attributes)
    attr_counts_B = Counter(B_attributes)

    # Prior: pooled count across both
    prior_counts = attr_counts_A + attr_counts_B

    # Total counts
    n_d1 = sum(attr_counts_A.values())
    n_d2 = sum(attr_counts_B.values())
    n_prior = sum(prior_counts.values())

    results = []
    for attr in prior_counts:
        c1 = attr_counts_A.get(attr, 0)
        c2 = attr_counts_B.get(attr, 0)
        prior = prior_counts[attr]

        # Dirichlet prior smoothing
        freq1 = (c1 + prior / n_prior) / (n_d1 + 1)
        freq2 = (c2 + prior / n_prior) / (n_d2 + 1)

        log_odds = np.log(freq1 / freq2)

        # Variance estimate for z-score
        var = 1 / (c1 + prior / n_prior) + 1 / (c2 + prior / n_prior)
        z = log_odds / np.sqrt(var)

        results.append({
            "attribute": attr,
            "A_count": c1,
            "B_count": c2,
            "log_odds": log_odds,
            "z_score": z
        })

    logodds_df = pd.DataFrame(results)
    # print(logodds_df)
    # Filter for attributes with enough evidence
    logodds_df["total_count"] = logodds_df["A_count"] + logodds_df["B_count"]
    logodds_df["A_freq"] = logodds_df["A_count"] / logodds_df["A_count"].sum()
    logodds_df["B_freq"] = logodds_df["B_count"] / logodds_df["B_count"].sum()
    logodds_df = logodds_df.sort_values(by="z_score", ascending=False).reset_index(drop=True)
    logodds_df["attribute_rank"] = logodds_df.index.tolist()
    logodds_df["attribute_rank"] = (logodds_df["attribute_rank"] / len(logodds_df) * 2 - 1) * -1

    # Normalize to [-1, 1], centered at 0
    max_abs = logodds_df["z_score"].abs().max()
    logodds_df["z_score_norm"] = logodds_df["z_score"] / max_abs
    logodds_df

    return logodds_df

translation_mapping = translation_mapping = {
    "pauvre": "poor",
    "beau": "handsome",
    "petit": "small",
    "jeune": "young",
    "joli": "pretty",
    "bon": "good",
    "femme": "woman",
    "charmant": "charming",
    "fille": "girl",
    "noble": "noble",
    "satisfait": "satisfied",
    "désolé": "sorry",
    "inspecteur": "inspector",
    "incapable": "incapable",
    "flic": "cop",
    "au courant de": "aware of",
    "en mesure de": "able of",
    # "en train de": "in the process of",
    "policier": "police officer",
    "certain": "certain",
    "sûr": "sure",
    # poss
    "monsieur": "gentleman",
    "père": "father",
    "mère": "mother",
    "mari": "husband",
    "fille": "daughter",
    "cœur": "heart",
    "enfant": "child",
    "frère": "brother",
    "oncle": "uncle",
    "âme": "soul",
    "pardessus": "overcoat",
    "collègue": "colleague",
    "revolver": "revolver",
    "interlocuteur": "interlocutor",
    "montre": "watch",
    "enquête": "investigation",
    "impression": "impression/feeling",
    "poche": "pocket",
    "pipe": "pipe",
    "bureau": "office",
    "victime": "victim",
    # agent
    "dire": "to say",
    "aimer": "to love",
    "vouloir": "to want",
    "mourir": "to die",
    "pleurer": "to cry",
    "répondre": "to answer",
    "parler": "to talk",
    "souffrir": "to suffer",
    "écrire": "to write",
    "devenir": "to become",
    "grommeler": "to grumble",
    "observer": "to observe",
    "téléphoner": "to make a call",
    "repérer": "to spot",
    "pénétrer": "to enter",
    "attraper": "to catch",
    "questionner": "to question",
    "raccrocher": "to hang up",
    "découvrir": "to discover",
    "noter": "to note"
}
def add_translation(row):
    fr_attribute = row['attribute']
    en_attribute = translation_mapping.get(fr_attribute, "unknown")
    return f"<b>{fr_attribute}</b><span style='font-size:85%;'><br><i>({en_attribute})</i></span>"

def get_most_discriminative_attributes(logodds_df, attribute_to_keep=10):
    # Get top and bottom attributes
    most_discriminative_attributes_df = pd.concat([
        logodds_df.sort_values('z_score', ascending=True).head(attribute_to_keep),
        logodds_df.sort_values('z_score', ascending=True).tail(attribute_to_keep)
    ])

    # Normalize z_score to [-1, 1]
    max_abs = most_discriminative_attributes_df["z_score"].abs().max()
    most_discriminative_attributes_df["z_score_norm"] = most_discriminative_attributes_df["z_score"] / max_abs

    # Sort for consistent display
    most_discriminative_attributes_df = most_discriminative_attributes_df.sort_values("z_score_norm", ascending=True).reset_index(drop=True)
    return most_discriminative_attributes_df

def generate_discriminative_attributes_fig(most_discriminative_attributes_df, attributes_type, show=True):
    # Create bar chart
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=most_discriminative_attributes_df["z_score_norm"],
        y=most_discriminative_attributes_df["translated_attributes"],
        orientation='h',
        width=0.99,  # controls bar thickness, range roughly from 0 to 1
        marker=dict(
            color=most_discriminative_attributes_df["z_score_norm"],
            colorscale="RdBu",
            cmin=-1,
            cmax=1,
            line=dict(width=0.1)  # 👈 removes the white border
        ),
        text=[f"{attribute}" for attribute in most_discriminative_attributes_df["translated_attributes"]],
        textposition="outside",  # ← label at the end of each bar
        insidetextanchor="start",
        showlegend=False,
        textfont=dict(
            size=50,
            family="Times Roman",
            color="black"
        )
    ))

    height=1400
    width=500

    # Update layout
    fig.update_layout(
        xaxis=dict(
            # title=dict(text="Normalized Z-Score (positive = more associated with detectives)",
            #            font=dict(size=18)),
            range=[-1.01, 1.01],
            tickfont=dict(size=24, family='Times Roman', color='black'),
            # zeroline=True,
            zerolinecolor='black',
            zerolinewidth=2
        ),
        # bargap=0,
        yaxis=dict(
            categoryorder='array',
            categoryarray=most_discriminative_attributes_df["translated_attributes"],
            visible=False,
            showticklabels=False,
            showgrid=False,
            zeroline=False
        ),
        template="simple_white",
        paper_bgcolor='rgba(0,0,0,0)',  # Transparent paper background
        plot_bgcolor='rgba(0,0,0,0)',   # Transparent plot area background
        height=height,
        width=width,
        margin=dict(l=80, r=160, t=0, b=50), # agent
    )

    fig.data[0].update(cliponaxis=False)  # try disabling clipping of text on axis

    if show:
        fig.show()
    fig.write_image(f"figures/{attributes_type}.png",
                    width=width,
                    height=height,
                    scale=4)

In [80]:
detectives_characters_dict = [char_dict for char_dict in overall_character_dataset if char_dict["label"] == 1]
non_detectives_characters_dict = [char_dict for char_dict in overall_character_dataset if char_dict["label"] == 0]

for attributes_type in ["agent_lemmas", "mod_lemmas", "pos_lemmas"]:
    logodds_df = get_z_score_df(characters_dict_A=detectives_characters_dict,
                                character_dicts_B=non_detectives_characters_dict,
                                attributes_type=attributes_type)

    most_discriminative_attributes_df = get_most_discriminative_attributes(logodds_df, attribute_to_keep=10)
    most_discriminative_attributes_df["translated_attributes"] = most_discriminative_attributes_df.apply(add_translation, axis=1)
    generate_discriminative_attributes_fig(most_discriminative_attributes_df, attributes_type, show=False)

In [88]:
detectives_characters_dict = [char_dict for char_dict in overall_character_dataset if char_dict["label"] == 1]
cluster_map = {0:"A",1:"C", 2:"B"}
for char_dict, cluster_label in zip(detectives_characters_dict, clusters_labels):
    char_dict["cluster"] = cluster_map[cluster_label]
char_dict.keys()

dict_keys(['file_name', 'char_name', 'char_id', 'mention_count', 'mention_ratio', 'attributes_count', 'plural_ratio', 'gender_ratio', 'agent_lemmas', 'patient_lemmas', 'mod_lemmas', 'pos_lemmas', 'mean_embedding', 'label', 'year', 'author', 'multidoc_alias', 'cluster'])

In [92]:
translation_mapping = {
    "agent_lemmas": [
        ('acquiescer', 'to nod / to agree'),
        ('affirmer', 'to assert'),
        ('aimer', 'to love'),
        ('ajouter', 'to add'),
        ('attraper', 'to catch'),
        ('attraper', 'to catch'),
        ('avaler', 'to swallow'),
        ('avoir', 'to have'),
        ('boire', 'to drink'),
        ('bourrer', 'to stuff'),
        ('connaître', 'to know'),
        ('crier', 'to shout'),
        ('croire', 'to believe'),
        ('croiser', 'to run into'),
        ('demeurer', 'to remain'),
        ('devenir', 'to become'),
        ('dire', 'to say'),
        ('dire', 'to say'),
        ('déclarer', 'to declare'),
        ('découvrir', 'to discover'),
        ('enfiler', 'to put on (clothes)'),
        ('foutre', 'to not care / to make fun of'),
        ('fumer', 'to smoke'),
        ('grommeler', 'to grumble'),
        ('interroger', 'to question'),
        ('interrompre', 'to interrupt'),
        ('lever', 'to raise / to lift'),
        ('mourir', 'to die'),
        ('murmurer', 'to murmur'),
        ('noter', 'to note'),
        ('observer', 'to observe'),
        ('parler', 'to talk'),
        ('permettre', 'to allow'),
        ('pleurer', 'to cry'),
        ('poser', 'to put down'),
        ('prier', 'to beg'),
        ('préférer', 'to prefer'),
        ('pénétrer', 'to enter'),
        ('questionner', 'to question'),
        ('raccrocher', 'to hang up'),
        ('regarder', 'to look'),
        ('reprendre', 'to resume'),
        ('repérer', 'to spot'),
        ('repérer', 'to spot / to notice'),
        ('répliquer', 'to reply'),
        ('répondre', 'to answer'),
        ("s'écrier", 'to exclaim'),
        ('savoir', 'to know'),
        ('se souvenir', 'to remember'),
        ('sentir', 'to feel'),
        ('sortir', 'to go out'),
        ('souffrir', 'to suffer'),
        ('soupirer', 'to sigh'),
        ('supposer', 'to suppose'),
        ('tenter', 'to attempt'),
        ('téléphoner', 'to make a call'),
        ('venir', 'to come'),
        ('vouloir', 'to want'),
        ('vérifier', 'to check'),
        ('écrire', 'to write')
    ],
    "mod_lemmas": [
        ('au courant de', 'aware of'),
        ('beau', 'handsome'),
        ('bon', 'good'),
        ('brave', 'brave'),
        ('censé', 'supposed'),
        ('certain', 'certain'),
        ('charmant', 'charming'),
        ('cher', 'dear'),
        ('commissaire', 'commissioner'),
        ('conscient', 'aware'),
        ('célèbre', 'famous'),
        ('digne', 'worthy'),
        ('désolé', 'sorry'),
        ('en mesure de', 'able of'),
        ('enquêteur', 'investigator'),
        ('excellent', 'excellent'),
        ('fatigué', 'tired'),
        ('faux', 'false / fake'),
        ('femme', 'woman'),
        ('fille', 'girl'),
        ('flic', 'cop'),
        ('gros', 'fat'),
        ('général', 'general'),
        ('honnête', 'honest'),
        ('incapable', 'incapable'),
        ('inspecteur', 'inspector'),
        ('ivre', 'drunk'),
        ('jaloux', 'jealous'),
        ('jeune', 'young'),
        ('joli', 'pretty'),
        ('large', 'broad / wide'),
        ('lourd', 'heavy'),
        ('malheureux', 'unhappy'),
        ('marié', 'married'),
        ('maussade', 'gloomy / sullen'),
        ('misérable', 'miserable'),
        ('mystérieux', 'mysterious'),
        ('noble', 'noble'),
        ('nu', 'naked'),
        ('pauvre', 'poor'),
        ('peine', 'sorrow / grief'),
        ('petit', 'small'),
        ('police', 'police'),
        ('policier', 'police officer'),
        ('proche', 'close / nearby'),
        ('responsable', 'responsible'),
        ('roux', 'red-haired'),
        ('satisfait', 'satisfied'),
        ('seul', 'alone'),
        ('spécial', 'special'),
        ('sûr', 'sure / confident'),
        ('turc', 'Turkish'),
        ('type', 'guy / dude'),
        ('vieux', 'old')
    ],
    "pos_lemmas": [
        ('ami', 'friend'),
        ('bureau', 'office / desk'),
        ('cabinet', 'office / cabinet'),
        ('chapeau', 'hat'),
        ('cheval', 'horse'),
        ('cheveu', 'hair'),
        ('client', 'client'),
        ('collaborateur', 'collaborator'),
        ('collègue', 'colleague'),
        ('compagnon', 'companion'),
        ('complice', 'accomplice'),
        ('corps', 'body'),
        ('cœur', 'heart'),
        ('devoir', 'duty'),
        ('dieu', 'god'),
        ('doigt', 'finger'),
        ('dos', 'back'),
        ('enfant', 'child'),
        ('ennemi', 'enemy'),
        ('enquête', 'investigation'),
        ('envie', 'desire'),
        ('femme', 'wife'),
        ('fille', 'daughter'),
        ('foi', 'faith'),
        ('frère', 'brother'),
        ('honneur', 'honor'),
        ('impression', 'impression / feeling'),
        ('inspecteur', 'inspector'),
        ('interlocuteur', 'interlocutor'),
        ('lunette', 'glasses'),
        ('mal', 'pain / suffering'),
        ('mari', 'husband'),
        ('maître', 'master'),
        ('monsieur', 'gentleman'),
        ('montre', 'watch'),
        ('mère', 'mother'),
        ('oncle', 'uncle'),
        ('ordre', 'order'),
        ('pardessus', 'overcoat'),
        ('parole', 'word / promise'),
        ('peau', 'skin'),
        ('pipe', 'pipe'),
        ('poche', 'pocket'),
        ('père', 'father'),
        ('revolver', 'revolver'),
        ('sac', 'bag'),
        ('téléphone', 'phone'),
        ('ventre', 'belly / stomach'),
        ('verre', 'glass / drink'),
        ('veste', 'jacket'),
        ('veston', 'jacket'),
        ('victime', 'victim'),
        ('visage', 'face'),
        ('voiture', 'car'),
        ('âme', 'soul')
    ]
}

def translate_attribute(fr_attr, mapping):
    en_attr = "unknown"
    for fr, en in mapping:
        if fr == fr_attr:
            en_attr = en
            break
    return f"{fr_attr} ({en_attr})"

In [99]:
all_attributes = []

most_discriminative_cluster_attributes_dict = {}
for target_cluster in ["A", "B", "C"]:
    cluster_characters_dict = [char_dict for char_dict in detectives_characters_dict if char_dict["cluster"] == target_cluster]
    non_cluster_characters_dict = [char_dict for char_dict in detectives_characters_dict if char_dict["cluster"] != target_cluster]

    most_discriminative_cluster_attributes_dict[target_cluster] = {}
    for attributes_type in ["agent_lemmas", "mod_lemmas", "pos_lemmas"]:
        logodds_df = get_z_score_df(characters_dict_A=cluster_characters_dict,
                                    character_dicts_B=non_cluster_characters_dict,
                                    attributes_type=attributes_type)

        attributes_list = logodds_df["attribute"].tolist()[:15]
        translation_map = translation_mapping[attributes_type]
        attributes_list = [translate_attribute(fr_attribute, translation_map) for fr_attribute in attributes_list]

        attributes_list = "\\\\".join(attributes_list)
        most_discriminative_cluster_attributes_dict[target_cluster][attributes_type] = attributes_list

most_discriminative_cluster_attributes_df = pd.DataFrame(most_discriminative_cluster_attributes_dict).T
most_discriminative_cluster_attributes_df.columns = ["Agent Verbs", "Modifiers", "Possessives"]
most_discriminative_cluster_attributes_df = most_discriminative_cluster_attributes_df.T
most_discriminative_cluster_attributes_df.columns = ["Cluster A", "Cluster B", "Cluster C"]

latex_code = most_discriminative_cluster_attributes_df.T.to_latex(index=True, escape=False)
print(latex_code)

\begin{tabular}{llll}
\toprule
 & Agent Verbs & Modifiers & Possessives \\
\midrule
Cluster A & dire (to say)\\s'écrier (to exclaim)\\répondre (to answer)\\déclarer (to declare)\\murmurer (to murmur)\\répliquer (to reply)\\interrompre (to interrupt)\\venir (to come)\\demeurer (to remain)\\reprendre (to resume)\\ajouter (to add)\\interroger (to question)\\affirmer (to assert)\\crier (to shout)\\prier (to beg) & cher (dear)\\jeune (young)\\brave (brave)\\malheureux (unhappy)\\vieux (old)\\pauvre (poor)\\excellent (excellent)\\faux (false / fake)\\digne (worthy)\\mystérieux (mysterious)\\général (general)\\bon (good)\\honnête (honest)\\misérable (miserable)\\célèbre (famous) & ami (friend)\\parole (word / promise)\\compagnon (companion)\\dieu (god)\\maître (master)\\monsieur (gentleman)\\revolver (revolver)\\ordre (order)\\foi (faith)\\cabinet (office / cabinet)\\ennemi (enemy)\\devoir (duty)\\complice (accomplice)\\cheval (horse)\\honneur (honor) \\
Cluster B & questionner (to question)\

In [100]:
def makecellify(cell):
    # Split on \n or \\ and rejoin with LaTeX line breaks
    lines = str(cell).split("\\")
    return r"\makecell{" + r"\\ ".join(lines) + "}"

# Apply formatting to every cell
for col in ["Cluster A", "Cluster B", "Cluster C"]:
    most_discriminative_cluster_attributes_df[col] = most_discriminative_cluster_attributes_df[col].map(makecellify)

# Convert to LaTeX
latex_code = most_discriminative_cluster_attributes_df.to_latex(index=True, escape=False)
latex_code = latex_code.replace("_", "\_").replace("\\\\ \\\\ ", "\\\\")
latex_code = latex_code.replace("Agent Verbs & ", "\\rotatebox{90}{\makecell{\\textbf{Agent Verbs}}} & ").replace("Possessives & ", "\midrule\n\\rotatebox{90}{\makecell{\\textbf{Possessives}}} & ").replace("Modifiers & ", "\midrule\n\\rotatebox{90}{\makecell{\\textbf{Modifiers}}} & ")
latex_code = latex_code.replace("Cluster A & Cluster B & Cluster C", "\\textbf{Cluster A} & \\textbf{Cluster B} & \\textbf{Cluster C}")
latex_code = latex_code.replace("{llll}", "{cccc}")
print(latex_code.replace("_", "\_"))

\begin{tabular}{cccc}
\toprule
 & \textbf{Cluster A} & \textbf{Cluster B} & \textbf{Cluster C} \\
\midrule
\rotatebox{90}{\makecell{\textbf{Agent Verbs}}} & \makecell{dire (to say)\\s'écrier (to exclaim)\\répondre (to answer)\\déclarer (to declare)\\murmurer (to murmur)\\répliquer (to reply)\\interrompre (to interrupt)\\venir (to come)\\demeurer (to remain)\\reprendre (to resume)\\ajouter (to add)\\interroger (to question)\\affirmer (to assert)\\crier (to shout)\\prier (to beg)} & \makecell{questionner (to question)\\téléphoner (to make a call)\\supposer (to suppose)\\regarder (to look)\\bourrer (to stuff)\\préférer (to prefer)\\fumer (to smoke)\\connaître (to know)\\boire (to drink)\\avoir (to have)\\se souvenir (to remember)\\savoir (to know)\\permettre (to allow)\\soupirer (to sigh)\\croire (to believe)} & \makecell{sentir (to feel)\\attraper (to catch)\\poser (to put down)\\foutre (to not care / to make fun of)\\repérer (to spot)\\tenter (to attempt)\\lever (to raise / to lift)\\cr